# Nearest-Neighbor Experiments

Here we want to test whether local similarity contains useful frad signals to improve `CatBoost` model. Three approaches will be compared:

- **kNN classifier** - fraud rate among closest transactions
- **Class-distance features** - distance to nearby fraud and legitemate transactions, then testing features with `CatBoost`.
- **Local Outlier Factor (LOF)** - how unusual a transaction to its neighbors.

We use the same stratified 5-fold split as our baseline. Predictions and neighbor features for validation rows are built **only from the training portion** of each fold.

## 1. Setup

`FAISS` is used for approximate neighboe searches.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

SEED = 42
N_SPLITS = 5
INNER_SPLITS = 3
K = 20
MAX_REFERENCE = 30_000   # Increase later if the results look promising
RUN_CATBOOST = True      # Set False to test the three neighbor methods first
TARGET = "is_fraud"

root = Path("../playground-series-s6e11")
train_df = pd.read_csv(root / "train.csv")
y = train_df[TARGET].astype(int).to_numpy()
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

baseline_file = Path("baseline_artifacts/baseline_oof.csv")
if not baseline_file.exists():
    baseline_file = Path("../02_establish_baseline/baseline_artifacts/baseline_oof.csv")
baseline = pd.read_csv(baseline_file)
baseline = train_df[["id"]].merge(baseline[["id", "fold", "oof_pred"]], on="id", how="left")
base_auc = roc_auc_score(y, baseline["oof_pred"])
print(f"Baseline OOF ROC-AUC: {base_auc:.6f}")
print("Fraud rate:", round(y.mean(), 4))

Baseline OOF ROC-AUC: 0.946094
Fraud rate: 0.0167


## 2. Build transactions as vectors

Since **Euclidian  distance** is sensetive to feature scaling, we need to standardize numeric columns and one-hot encode categorical columns. Boolean variables are treated as 0/1.

In [8]:
features = [c for c in train_df.columns if c not in ["id", TARGET]]
cat_cols = train_df[features].select_dtypes(include=["object", "string", "category"]).columns.tolist()
bool_cols = train_df[features].select_dtypes(include=["bool", "boolean"]).columns.tolist()
num_cols = [c for c in features if c not in cat_cols + bool_cols]

numeric = train_df[num_cols].astype("float32").fillna(train_df[num_cols].median())
encoded = pd.get_dummies(
    train_df[cat_cols].fillna("__MISSING__").astype(str), dtype="float32"
)
boolean = train_df[bool_cols].astype("float32").fillna(0)

num_array = numeric.to_numpy(dtype="float32")
other_array = np.column_stack([encoded.to_numpy(), boolean.to_numpy()]).astype("float32")

# Original CatBoost input, with its own categorical preprocessing
X_cb = train_df[features].copy()
for c in cat_cols:
    X_cb[c] = X_cb[c].astype("string").fillna("__MISSING__").astype(str)
for c in bool_cols:
    X_cb[c] = X_cb[c].astype("float32")

print("Vector dimension:", num_array.shape[1] + other_array.shape[1])
print("Categorical:", cat_cols)

Vector dimension: 54
Categorical: ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']


## 3. Neighbor features

The reference set is a **random**, **unlabeled-in-selection sample** of the training split (we NEVER sample using target labels). Labels are used only after that sample is chosen.

We use **Hierarchical Navigable Small World(HNSW)** index for approximate nearest neighbor. Each feature function fits its scaler and indices only on the supplied reference rows. For **local reachability density LOF**, we compute local reachability densities from the reference set, then score unseen query rows (novelty-style scoring).

In [9]:
neighbor_cols = [
    "knn_fraud_rate", "knn_mean_distance",
    "distance_to_fraud", "distance_to_normal", "distance_gap",
    "lof_score",
]


def vectors(reference_idx, query_idx):
    scaler = StandardScaler()
    a = scaler.fit_transform(num_array[reference_idx]).astype("float32")
    b = scaler.transform(num_array[query_idx]).astype("float32")
    # Avoid extreme numeric values dominating the distance
    a = np.clip(a, -5, 5)
    b = np.clip(b, -5, 5)
    return (
        np.ascontiguousarray(np.column_stack([a, other_array[reference_idx]])),
        np.ascontiguousarray(np.column_stack([b, other_array[query_idx]])),
    )


def make_index(points):
    index = faiss.IndexHNSWFlat(points.shape[1], 24)
    index.hnsw.efConstruction = 80
    index.hnsw.efSearch = 64
    index.add(points)
    return index


def search(index, points, k):
    # FAISS returns squared L2 distances
    d2, idx = index.search(points, k)
    return np.sqrt(np.maximum(d2, 0)), idx


def neighbor_features(train_idx, query_idx, seed=SEED):
    rng = np.random.default_rng(seed)
    ref_idx = rng.choice(train_idx, size=min(MAX_REFERENCE, len(train_idx)), replace=False)
    ref, query = vectors(ref_idx, query_idx)
    labels = y[ref_idx]

    index = make_index(ref)
    ref_d, ref_neighbors = search(index, ref, K + 1)
    # The nearest neighbor of each reference point is itself
    ref_d = ref_d[:, 1:]
    ref_neighbors = ref_neighbors[:, 1:]
    k_distance = ref_d[:, -1]
    reach_ref = np.maximum(ref_d, k_distance[ref_neighbors])
    lrd_ref = 1 / np.maximum(reach_ref.mean(axis=1), 1e-8)

    fraud_index = make_index(ref[labels == 1])
    normal_index = make_index(ref[labels == 0])

    result = np.empty((len(query_idx), len(neighbor_cols)), dtype="float32")
    batch_size = 10_000
    for start in range(0, len(query_idx), batch_size):
        end = min(start + batch_size, len(query_idx))
        q = query[start:end]
        d, neighbors = search(index, q, K)
        d_fraud, _ = search(fraud_index, q, 5)
        d_normal, _ = search(normal_index, q, 5)

        query_lrd = 1 / np.maximum(
            np.maximum(d, k_distance[neighbors]).mean(axis=1), 1e-8
        )
        lof = lrd_ref[neighbors].mean(axis=1) / query_lrd

        result[start:end] = np.column_stack([
            labels[neighbors].mean(axis=1),
            d.mean(axis=1),
            d_fraud.mean(axis=1),
            d_normal.mean(axis=1),
            d_normal.mean(axis=1) - d_fraud.mean(axis=1),
            lof,
        ])
    return pd.DataFrame(result, columns=neighbor_cols, index=query_idx)

# Experiment 01 - kNN classifier

A simple neighbor-based prediction is the fraction of fraud cases among the nearest `K` transactions. We also evaluate class distances and LOF **without training `CatBoost`**. Unlike absolute distance or LOF, higher `distance_gap` means closer to fraud than normal.

We use outer OOF predictions for all three methods, so each scored transaction is excluded from the reference set.

In [10]:
neighbor_oof = pd.DataFrame(np.nan, index=np.arange(len(train_df)), columns=neighbor_cols)
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(num_array, y), start=1):
    f = neighbor_features(train_idx, valid_idx, seed=SEED + fold)
    neighbor_oof.iloc[valid_idx] = f.to_numpy()
    fold_scores.append({
        "fold": fold,
        "knn_auc": roc_auc_score(y[valid_idx], f["knn_fraud_rate"]),
        "distance_auc": roc_auc_score(y[valid_idx], f["distance_gap"]),
        "lof_auc": roc_auc_score(y[valid_idx], f["lof_score"]),
    })
    print(f"Fold {fold}: kNN={fold_scores[-1]['knn_auc']:.6f}, "
          f"distance={fold_scores[-1]['distance_auc']:.6f}, "
          f"LOF={fold_scores[-1]['lof_auc']:.6f}")

individual_scores = pd.Series({
    col: roc_auc_score(y, neighbor_oof[col]) for col in neighbor_cols
}).sort_values(ascending=False)
print("\nIndividual OOF ROC-AUC:")
display(individual_scores.to_frame("auc"))
display(pd.DataFrame(fold_scores))

Fold 1: kNN=0.747470, distance=0.858166, LOF=0.696261
Fold 2: kNN=0.753379, distance=0.863247, LOF=0.693071
Fold 3: kNN=0.749372, distance=0.870501, LOF=0.699070
Fold 4: kNN=0.749362, distance=0.859711, LOF=0.656338
Fold 5: kNN=0.745250, distance=0.857311, LOF=0.700979

Individual OOF ROC-AUC:


,auc
distance_gap,0.861748
distance_to_normal,0.798978
knn_mean_distance,0.796217
knn_fraud_rate,0.748966
lof_score,0.689035
distance_to_fraud,0.503713


,fold,knn_auc,distance_auc,lof_auc
0,1,0.747470,0.858166,0.696261
1,2,0.753379,0.863247,0.693071
2,3,0.749372,0.870501,0.699070
3,4,0.749362,0.859711,0.656338
4,5,0.745250,0.857311,0.700979


## Experiment 02 — Neighbor Features with CatBoost

The class-distance gap is the strongest standalone neighbor feature, reaching an OOF `ROC-AUC` of **0.861748**.

Next, we will add all six neighbor features to CatBoost to see whether local similarity provides information beyond the original features.

We use inner cross-validation to generate training features without using each transaction's own label.

In [11]:
cb_oof = np.full(len(train_df), np.nan)
cb_fold_results = []

clf_params = dict(
    iterations=3000, depth=6, learning_rate=0.05,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=SEED, verbose=False, thread_count=-1,
)

if RUN_CATBOOST:
    for fold, (train_idx, valid_idx) in enumerate(cv.split(num_array, y), start=1):
        inner_cv = StratifiedKFold(
            n_splits=INNER_SPLITS, shuffle=True, random_state=SEED
        )
        train_neighbor = pd.DataFrame(
            np.nan, index=train_idx, columns=neighbor_cols
        )
        for inner_fold, (inner_train, inner_valid) in enumerate(
            inner_cv.split(train_idx, y[train_idx]), start=1
        ):
            inner_features = neighbor_features(
                train_idx[inner_train], train_idx[inner_valid],
                seed=SEED + 100 * fold + inner_fold,
            )
            train_neighbor.loc[inner_features.index] = inner_features

        # Reuse outer-fold OOF features computed in Experiment A
        valid_neighbor = neighbor_oof.iloc[valid_idx]
        X_train = X_cb.iloc[train_idx].copy()
        X_valid = X_cb.iloc[valid_idx].copy()
        for col in neighbor_cols:
            X_train[col] = train_neighbor.loc[train_idx, col].to_numpy()
            X_valid[col] = valid_neighbor[col].to_numpy()

        model = CatBoostClassifier(**clf_params)
        model.fit(
            X_train, y[train_idx], cat_features=cat_cols,
            eval_set=(X_valid, y[valid_idx]),
            early_stopping_rounds=100, use_best_model=True,
        )
        pred = model.predict_proba(X_valid)[:, 1]
        cb_oof[valid_idx] = pred
        baseline_fold = roc_auc_score(y[valid_idx], baseline["oof_pred"].iloc[valid_idx])
        new_fold = roc_auc_score(y[valid_idx], pred)
        cb_fold_results.append({
            "fold": fold, "baseline_auc": baseline_fold,
            "knn_features_auc": new_fold, "delta": new_fold - baseline_fold,
        })
        print(f"Fold {fold}: AUC={new_fold:.6f}, delta={new_fold-baseline_fold:+.6f}")

Fold 1: AUC=0.943180, delta=-0.001382
Fold 2: AUC=0.947022, delta=-0.000441
Fold 3: AUC=0.949920, delta=-0.000632
Fold 4: AUC=0.945218, delta=-0.001134
Fold 5: AUC=0.941018, delta=-0.000873


## 4. Results

We will compare the standalone neighbor-based approaches with our original `CatBoost` baseline and `CatBoost` trained with additional neighbor features.

In [12]:
comparison = pd.DataFrame([
    {"approach": "Original CatBoost", "oof_auc": base_auc},
    {"approach": "kNN classifier (fraud fraction)",
     "oof_auc": roc_auc_score(y, neighbor_oof["knn_fraud_rate"])},
    {"approach": "Class-distance gap", 
     "oof_auc": roc_auc_score(y, neighbor_oof["distance_gap"])},
    {"approach": "LOF anomaly score",
     "oof_auc": roc_auc_score(y, neighbor_oof["lof_score"])},
])
if RUN_CATBOOST:
    comparison.loc[len(comparison)] = {
        "approach": "CatBoost + neighbor features",
        "oof_auc": roc_auc_score(y, cb_oof),
    }
comparison["delta_vs_baseline"] = comparison["oof_auc"] - base_auc
display(comparison.sort_values("oof_auc", ascending=False))
if RUN_CATBOOST:
    display(pd.DataFrame(cb_fold_results))

,approach,oof_auc,delta_vs_baseline
0,Original CatBoost,0.946094,0.000000
4,CatBoost + neighbor features,0.945242,-0.000852
2,Class-distance gap,0.861748,-0.084347
1,kNN classifier (fraud fraction),0.748966,-0.197128
3,LOF anomaly score,0.689035,-0.257059


,fold,baseline_auc,knn_features_auc,delta
0,1,0.944562,0.943180,-0.001382
1,2,0.947463,0.947022,-0.000441
2,3,0.950552,0.949920,-0.000632
3,4,0.946352,0.945218,-0.001134
4,5,0.941891,0.941018,-0.000873


## 5. Findings

The class-distance gap was the strongest standalone neighbor feature, achieving an OOF `ROC-AUC` of **0.861748**.

Distance to normal transactions was more informative than distance to fraud transactions. This suggests that fraudulent transactions may be easier to identify by how different they are from normal behavior.

The kNN classifier achieved **0.748966**, while LOF reached **0.689035**. Both detected some useful structure, but performed well below `CatBoost`.

Adding all six neighbor features to CatBoost decreased OOF ROC-AUC from **0.946094 to 0.945242**, with lower scores on every fold.

We also tested `distance_gap` alone using a reference set of ***30,000*** transactions. It did not improve CatBoost on any of the five folds. Increasing the reference set to ***100,000*** showed no improvement on the first two folds, so we stopped that experiment early.

**Conclusion:** Neighbor-based methods reveal useful structure in the data, but none of our tested approaches improved `CatBoost`.